In [1]:
# 📌 1. Import Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample
import pickle

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lakha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv(r"C:\Users\Lakha\Downloads\fake_job_postings.csv\fake_job_postings.csv")
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [3]:
df = df[['description', 'fraudulent']]
df.dropna(inplace=True)
df.head()

,description,fraudulent
0,"Food52, a fast-growing, James Beard Award-winn...",0
1,Organised - Focused - Vibrant - Awesome!Do you...,0
2,"Our client, located in Houston, is actively se...",0
3,THE COMPANY: ESRI – Environmental Systems Rese...,0
4,JOB TITLE: Itemization Review ManagerLOCATION:...,0


In [4]:
# 3. Preprocessing Function
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()                             # 1. Lowercase
    text = re.sub(r"http\S+|www\S+", "", text)           # 2. Remove URLs
    text = re.sub(r"<.*?>", "", text)                    # 3. Remove HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)                # 4. Remove punctuation/special chars
    text = re.sub(r"\d+", "", text)                      # 5. Remove numbers
    tokens = text.split()                                # 6. Tokenization
    tokens = [w for w in tokens if w not in stop_words]  # 7. Stopword removal
    tokens = [stemmer.stem(w) for w in tokens]           # 8. Stemming
    tokens = [w for w in tokens if len(w) > 2]           # 9. Remove short words
    text = " ".join(tokens)                              # 10. Normalize text
    return text

df["cleaned"] = df["description"].apply(clean_text)
df.head()

,description,fraudulent,cleaned
0,"Food52, a fast-growing, James Beard Award-winn...",0,food fast grow jame beard award win onlin food...
1,Organised - Focused - Vibrant - Awesome!Do you...,0,organis focus vibrant awesom passion custom se...
2,"Our client, located in Houston, is actively se...",0,client locat houston activ seek experienc comm...
3,THE COMPANY: ESRI – Environmental Systems Rese...,0,compani esri environment system research insti...
4,JOB TITLE: Itemization Review ManagerLOCATION:...,0,job titl item review managerloc fort worth dep...


In [5]:
#4. Balance Dataset (Fixes "Always Real" Problem)
df_majority = df[df.fraudulent == 0]   # REAL jobs
df_minority = df[df.fraudulent == 1]   # FAKE jobs

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42)

print("Balanced dataset size:", df_balanced.shape)
print(df_balanced['fraudulent'].value_counts())

Balanced dataset size: (34028, 3)
fraudulent
1    17014
0    17014
Name: count, dtype: int64


In [6]:
#5. Feature Extraction (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df_balanced['cleaned'])
y = df_balanced['fraudulent']

In [7]:
# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
# 7. Train Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [9]:
# 8. Evaluate Model
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9685571554510726
Precision: 0.953859299344916
Recall: 0.9847103793002058
F1 Score: 0.9690393518518519
Confusion Matrix:
 [[3243  162]
 [  52 3349]]


In [10]:
# 9. Save Model + TF-IDF for Flask
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

In [11]:
# 10. Test Prediction (Demo)
sample = ["Work from home job, earn 50,000 per week, no experience needed"]
cleaned = clean_text(sample[0])
vector = tfidf.transform([cleaned])
print("Prediction:", "FAKE" if model.predict(vector)[0] == 1 else "REAL")

Prediction: FAKE
